# Student Social Network Segmentation — K-Means Clustering

**Author:** Dev Shah  
**Domain:** Education Analytics · Unsupervised Learning · Clustering  
**Tools:** Python · scikit-learn · scipy · pandas · seaborn · matplotlib

---

## Quick Summary

| Item | Value |
|---|---|
| Dataset | 15,000 student social network keyword profiles |
| Algorithm | K-Means Clustering (K=4) |
| Dimensionality Reduction | PCA (for visualisation only) |
| Silhouette Score | Moderate (expected for sparse high-dim data) |
| Clusters Found | 4 distinct student personas |

## Clusters Discovered

| Cluster | Label | Size | Priority |
|---|---|---|---|
| 0 | 👗 Fashion-Forward | ~432 (3%) | Low-Medium |
| 1 | 🤫 Silent Majority | ~10,520 (70%) | Medium |
| 2 | 🏈🎵 Actively Engaged | ~3,517 (23%) | Low |
| 3 | ⚠️ At-Risk Group | ~528 (4%) | **HIGH** |

---

# Project: Segmenting Students Based on Their Social Network Profiles

High-school students constantly express their interests, personalities, and social behaviours through the words they use online.  
This project analyses a dataset of keyword mentions from **15,000 student social network profiles** to discover natural groupings — without any pre-defined labels.

---

## Project Objective

By performing unsupervised clustering on this dataset, we aim to:
- **Identify underlying behaviour patterns** among students
- **Understand which groups** are inclined toward sports, appearance, music, shopping, religion, or risky behaviours
- **Explore** how students with similar online expression naturally cluster together
- **Demonstrate** how unsupervised learning turns raw keyword data into actionable insights

In simple terms — we let the data speak for itself.

---

## Dataset Overview

| Category | Features |
|---|---|
| Demographics & Network | `gradyear`, `gender`, `age`, `NumberOffriends` |
| Sports | `basketball`, `football`, `soccer`, `softball`, `volleyball`, `swimming`, `cheerleading`, `baseball`, `tennis`, `sports` |
| Appearance / Personality | `cute`, `sex`, `sexy`, `hot`, `hair`, `dress`, `blonde`, `clothes` |
| Social Life & Activities | `kissed`, `dance`, `band`, `marching`, `music`, `rock`, `mall`, `shopping` |
| Religion | `god`, `church`, `jesus`, `bible` |
| Brand Mentions | `hollister`, `abercrombie` |
| Negative / Risk-Prone | `die`, `death`, `drunk`, `drugs` |

---

**Next Step:** Load the dataset and inspect its structure before any preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

## Step 1: Data Loading & Exploration

We begin by loading the CSV and performing a basic health check:
1. Preview the data
2. Check shape and data types
3. Identify missing values and their percentage
4. Check for duplicate rows

This gives us a clear picture of what cleaning is needed before we can cluster.

In [ ]:
df = pd.read_csv('Clustering_Marketing.csv')

print('Preview of the dataset:')
df.head()

In [ ]:
print('Dataset Shape:', df.shape)
print('\nDataset Info:')
df.info()

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({'Missing Count': missing_count, 'Missing %': missing_pct})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]

print('Columns with Missing Values:')
display(missing_summary)

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('\nBasic Statistics (numeric columns):')
df.describe().T

## Step 2: Data Cleaning

Based on the exploration above, here is our cleaning plan:

**1. Drop `gender` and `gradyear`**
- `gender` is categorical and has ~8.9% missing values. Encoding and imputing would introduce noise since gender is not a keyword-behaviour signal.
- `gradyear` is an ordinal class identifier (graduation year) — it doesn't reflect how a student expresses themselves online.

**2. Convert `age` to numeric**
- `age` was loaded as `object` dtype; we convert it using `pd.to_numeric`.

**3. Cap and Impute missing `age`**
- Ages outside the realistic high-school range (13–20) are replaced with `NaN` and then filled with the **median** to avoid outlier distortion.

**4. Remove extreme outlier rows**
- We check each student's total keyword mention count. One student had 401 total mentions (next closest: 127) — a clear extreme outlier that would dominate clustering. We remove rows where total keyword sum > 100.

**5. All keyword columns**
- Already clean integers — no action needed.

In [ ]:
# Drop categorical / non-behavioural columns
df_clean = df.drop(['gradyear', 'gender'], axis=1)

# Convert age to numeric
df_clean['age'] = pd.to_numeric(df_clean['age'], errors='coerce')

# Cap unrealistic ages and impute with median
median_age = df_clean['age'].median()
df_clean.loc[(df_clean['age'] < 13) | (df_clean['age'] > 20), 'age'] = np.nan
df_clean['age'] = df_clean['age'].fillna(median_age)

print('Age after cleaning:')
print(f'  Min: {df_clean["age"].min():.1f}, Max: {df_clean["age"].max():.1f}, Missing: {df_clean["age"].isnull().sum()}')

In [ ]:
# Identify extreme outlier students by total keyword mention count
keyword_cols = list(df_clean.columns[2:])  # all columns after age and NumberOffriends
row_sums = df_clean[keyword_cols].sum(axis=1)

print('Total keyword mention count statistics:')
print(row_sums.describe().round(2))
print(f'\nTop 5 most active students (by total mentions):')
print(row_sums.nlargest(5))

In [ ]:
# Remove extreme outlier (total keyword sum > 100)
before = df_clean.shape[0]
df_clean = df_clean[row_sums <= 100].copy().reset_index(drop=True)
after = df_clean.shape[0]

print(f'Rows before outlier removal: {before}')
print(f'Rows after  outlier removal: {after}')
print(f'Removed: {before - after} extreme outlier row(s)')
print(f'\nCleaned dataset shape: {df_clean.shape}')

print('\nRemaining missing values:')
print(df_clean.isnull().sum().sum(), '(total across all columns)')

## Step 3: Standardize Features

Clustering algorithms like K-Means rely on **Euclidean distance** between data points.  
If one feature has a much larger range (e.g., `NumberOffriends` can go up to 300+) while others are small counts (0–10), the larger-magnitude feature will dominate the distance calculation unfairly.

**StandardScaler** transforms each feature so that:
- **Mean = 0**
- **Standard Deviation = 1**

This ensures every keyword — whether `basketball` or `drugs` — contributes equally to the clustering decision.

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_clean)

df_scaled = pd.DataFrame(scaled_data, columns=df_clean.columns)

print('Shape of standardized dataset:', df_scaled.shape)
print('\nVerification — Mean ≈ 0, Std ≈ 1:')
display(df_scaled.describe().loc[['mean', 'std']].round(4))

---

## What is Clustering?

**Clustering** is an unsupervised learning technique that groups data points such that:
- Points **within the same cluster** are as similar as possible (low intra-cluster distance)
- Points **across different clusters** are as different as possible (high inter-cluster distance)

There are no labels — the algorithm discovers the structure entirely from the data.

### Why Clustering for This Project?

We have ~15,000 student profiles, each described by 38 keyword counts.  
We don't know upfront how many student "types" exist.  
Clustering lets the data reveal these natural groupings — who is the athlete, the fashionista, the at-risk student — without us telling it what to look for.

---

### The Two Methods We Will Explore

| Method | How It Works | Strength |
|---|---|---|
| **Hierarchical Clustering** | Builds a tree (dendrogram) of merges based on distance | No need to pre-specify K |
| **K-Means Clustering** | Assigns each point to the nearest centroid, iteratively optimises | Fast, scalable, clear centroids |

We will run both and explain which is more appropriate for this project.

## Step 4: Hierarchical Clustering (Exploratory)

Hierarchical Clustering builds a **dendrogram** — a tree that shows at what distance data points merge together.  
We sample **2,000 students** for this step because hierarchical clustering has **O(n²) memory complexity** — running it on all 15,000 rows would require enormous memory and time.

We test multiple linkage methods and evaluate each with a **Silhouette Score**.

| Silhouette Score | Interpretation |
|---|---|
| > 0.70 | Excellent |
| 0.40–0.70 | Good |
| 0.25–0.40 | Weak |
| < 0.25 | Bad |

In [ ]:
# Sample 2000 rows for hierarchical clustering (O(n^2) memory constraint)
np.random.seed(42)
sample_idx = np.random.choice(len(df_scaled), size=2000, replace=False)
df_sample = df_scaled.iloc[sample_idx].reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, method in zip(axes, ['ward', 'complete', 'average']):
    Z = linkage(df_sample, method=method, metric='euclidean')
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, no_labels=True)
    ax.set_title(f'Dendrogram — {method.capitalize()} Linkage', fontsize=12)
    ax.set_xlabel('Data Points (truncated, n=2000)')
    ax.set_ylabel('Euclidean Distance')

plt.suptitle('Hierarchical Clustering Dendrograms (Sample of 2,000 Students)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('Silhouette Scores — Hierarchical Clustering (3 clusters, n=2000):')
sil_results = {}
for method in ['ward', 'complete', 'average', 'single']:
    Z = linkage(df_sample, method=method, metric='euclidean')
    labels_h = fcluster(Z, t=3, criterion='maxclust')
    sil = silhouette_score(df_sample, labels_h)
    sil_results[method] = round(sil, 4)
    print(f'  {method:10s} linkage → Silhouette Score: {sil:.4f}')

### ⚠️ Why Hierarchical Clustering is NOT Worth Following for This Project

Although hierarchical clustering gave us useful visual intuition via dendrograms, **it is not the appropriate primary method here**. Here is why:

| Limitation | Explanation |
|---|---|
| **Scalability Problem** | Hierarchical clustering has **O(n²) time and memory complexity**. With 15,000 students × 38 features, it becomes computationally infeasible on the full dataset — we had to sample down to 2,000 rows just to run it. |
| **No Re-assignment** | Once two points are merged in the tree, they are permanently grouped. K-Means iteratively reassigns every point to the closest centroid, producing globally better-optimised clusters. |
| **Weak Silhouette Scores on Full Data** | Even on the sample, scores are in the 0.10–0.30 range — the cluster separation is weak, as expected from a high-dimensional sparse keyword dataset. K-Means produces more compact and better-separated clusters. |
| **Hard to Interpret at Scale** | A dendrogram for 15,000 points is unreadable. K-Means centroids and bar-plot profiles are far more actionable for business use. |
| **No Clear Cut-off** | The dendrograms don't show an obvious "gap" to cut at — selecting K remains guesswork, whereas K-Means paired with the Elbow Method gives a data-driven decision. |

**Conclusion → We proceed with K-Means Clustering as our primary method.**

## Step 5: Elbow Method — Finding the Optimal Number of Clusters (K)

K-Means requires us to specify K upfront.  
The **Elbow Method** plots **WCSS (Within-Cluster Sum of Squares)** against different values of K.

- As K increases, WCSS decreases — more clusters always fit the data better.
- The **"elbow"** is where this improvement starts to diminish sharply.
- We pick the K at the bend — beyond it, adding more clusters gives minimal gain.

$$WCSS = \sum_{k=1}^{K} \sum_{x_i \in C_k} ||x_i - \mu_k||^2$$

Where $\mu_k$ is the centroid of cluster $C_k$.

In [ ]:
wcss = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(K_range), wcss, marker='o', linewidth=2, color='steelblue', markerfacecolor='red')
plt.title('The Elbow Method — Optimal K for K-Means', fontsize=14)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.xticks(list(K_range))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print('WCSS per K:')
for k, w in zip(K_range, wcss):
    print(f'  K={k}: {w:,.2f}')

From the elbow plot, the curve shows the steepest drop at K=2→3 and begins to flatten noticeably after **K=4**.  
Choosing K=4 gives us:
- A meaningful reduction in WCSS over K=3
- A richer set of student segments for business interpretation
- Reasonable cluster sizes (no tiny clusters)

We proceed with **K = 4** as our optimal number of clusters.

## Step 6: Fit the K-Means Model

We now fit K-Means with **K = 4** on the full standardized dataset.

Key parameters:
- `n_clusters=4` — number of groups
- `random_state=42` — reproducible results
- `n_init=10` — runs 10 times with different centroid initializations, picks the best (lowest WCSS)

In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans.fit(df_scaled)

print(f'K-Means model fitted with K = {optimal_k}')
print(f'Final WCSS (Inertia): {kmeans.inertia_:,.2f}')

In [ ]:
df_clean['Cluster'] = kmeans.labels_

print('Cluster Distribution:')
display(df_clean['Cluster'].value_counts().sort_index().rename('Count').to_frame())

print('\nCluster Proportions (%):')
display((df_clean['Cluster'].value_counts(normalize=True).sort_index() * 100).round(2).rename('Proportion %').to_frame())

In [ ]:
sil_km = silhouette_score(df_scaled, kmeans.labels_)
print(f'K-Means Silhouette Score (K=4): {sil_km:.4f}')
print()
print('Silhouette Score Reference:')
print('  > 0.70   → Excellent')
print('  0.40–0.70 → Good')
print('  0.25–0.40 → Weak but interpretable')
print('  < 0.25   → Poor')
print()
print('Note: A moderate silhouette score is expected for high-dimensional, sparse keyword')
print('count data. The clusters are still meaningful and interpretable from their profiles.')

## Step 7: PCA Visualization

Our dataset has **38 features** — impossible to visualise in raw form.  
**Principal Component Analysis (PCA)** compresses the high-dimensional feature space into 2 principal components (PC1 and PC2) that capture the maximum variance.

We then plot every student as a dot in this 2D space, **colored by their K-Means cluster**, to visually verify that the clusters have spatial separation.

> **Important:** PCA does NOT influence the clustering — it is used only for visualization.  
> The K-Means algorithm ran on all 38 standardized features, not on the PCA-reduced data.

### How PCA Works

PCA finds new axes (principal components) that are:
1. **Linear combinations** of the original features
2. **Orthogonal** (uncorrelated with each other)
3. Ordered so that **PC1 captures the most variance**, PC2 the second most, and so on

By projecting onto PC1 and PC2, we get the best possible 2D view of cluster separation.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(df_scaled)

explained = pca.explained_variance_ratio_
print(f'PC1 explains: {explained[0]*100:.2f}% of variance')
print(f'PC2 explains: {explained[1]*100:.2f}% of variance')
print(f'Total variance captured by 2 components: {sum(explained)*100:.2f}%')

In [ ]:
plt.figure(figsize=(10, 7))

palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=kmeans.labels_,
    palette=palette,
    alpha=0.5,
    s=15,
    legend='full'
)

plt.title('K-Means Clusters — PCA 2D Projection', fontsize=14)
plt.xlabel(f'PC1 ({explained[0]*100:.1f}% Variance Explained)')
plt.ylabel(f'PC2 ({explained[1]*100:.1f}% Variance Explained)')
plt.legend(title='Cluster', loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
pca_full = PCA(random_state=42)
pca_full.fit(df_scaled)
cumulative_var = pca_full.explained_variance_ratio_.cumsum()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, marker='o', color='steelblue')
plt.axhline(y=0.80, color='red', linestyle='--', label='80% Threshold')
plt.title('Cumulative Explained Variance — All Principal Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

n_80 = int(np.argmax(cumulative_var >= 0.80)) + 1
print(f'Number of components needed to explain 80% of variance: {n_80}')
print(f'Our dataset has 38 features — {n_80} PCs are enough to represent 80% of the information.')

## Step 8: Extract Cluster Profiles

Now we understand what each cluster looks like by computing the **mean value of each feature per cluster** on the original (un-scaled) data.  
This reveals the dominant interests and behaviour patterns of each group.

In [ ]:
profile = df_clean.groupby('Cluster')[keyword_cols].mean().round(3)

print('Cluster Profile — Mean Feature Values per Cluster:')
display(profile.T)

In [ ]:
categories = {
    'Sports Mentions': ['basketball', 'football', 'soccer', 'softball', 'volleyball', 'swimming', 'cheerleading', 'baseball', 'tennis', 'sports'],
    'Appearance & Personality': ['cute', 'sex', 'sexy', 'hot', 'hair', 'dress', 'blonde', 'clothes'],
    'Social Life & Activities': ['kissed', 'dance', 'band', 'music', 'rock', 'mall', 'shopping'],
    'Religion': ['god', 'church', 'jesus', 'bible'],
    'Fashion Brands': ['hollister', 'abercrombie'],
    'Negative / Risk-Prone': ['die', 'death', 'drunk', 'drugs']
}

for cat_name, cols in categories.items():
    n = len(cols)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    fig.suptitle(f'Cluster Profiles — {cat_name}', fontsize=13, fontweight='bold')
    for ax, col in zip(axes, cols):
        sns.barplot(x=df_clean['Cluster'], y=df_clean[col], errorbar=None,
                    palette=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], ax=ax)
        ax.set_title(col, fontsize=10)
        ax.set_xlabel('Cluster')
        ax.set_ylabel('Mean Count')
    plt.tight_layout()
    plt.show()

## Step 9: Interpret the Clusters

Based on the cluster profiles and bar plots above, we can assign a clear identity to each cluster.

---

### Cluster 0 — 👗 The Fashion & Brand-Conscious
**Size:** ~432 students (~3%)  
**Key Signal:** Very high `hollister`, `abercrombie`, `shopping`, `hair`, `mall`, `clothes`, `cute`.  
This group's online identity revolves around fashion brands and appearance — the classic "mall culture" segment. Their mentions of risky or sports keywords are minimal; their world is entirely appearance and consumer brands.

---

### Cluster 1 — 🤫 The Silent Majority (Passive / Low-Activity)
**Size:** ~10,520 students (~70%)  
**Key Signal:** Consistently low mention counts across ALL keyword categories.  
These students barely express themselves online. They are the "background noise" of the dataset — no single dominant interest signal. This is the largest group by far.

---

### Cluster 2 — 🏈🎵 The Actively Engaged Social Students
**Size:** ~3,517 students (~23%)  
**Key Signal:** Elevated across sports (`basketball`, `football`, `sports`), `music`, `dance`, `god`, `church`, `shopping`.  
Well-rounded, socially active students involved in multiple areas of school life — sports, music, faith, and social outings.

---

### Cluster 3 — ⚠️ The High-Engagement, At-Risk Group
**Size:** ~528 students (~4%)  
**Key Signal:** Extremely high `hair`, `sex`, `kissed`, `music`, `rock`, `drunk`, `drugs`, `die`, `death`.  
This is the most active group online — but their expression is dominated by risky and concerning themes alongside appearance and social mentions. This cluster is the highest welfare priority.

---

| Cluster | Label | Size | % | Core Theme |
|---|---|---|---|---|
| 0 | Fashion-Forward | ~432 | 3% | Brands, shopping, appearance |
| 1 | Silent Majority | ~10,520 | 70% | Low/no online expression |
| 2 | Actively Engaged | ~3,517 | 23% | Sports, music, social, faith |
| 3 | At-Risk Group | ~528 | 4% | High engagement + risky keywords |

In [ ]:
cluster_labels = {
    0: 'Fashion-Forward',
    1: 'Silent Majority',
    2: 'Actively Engaged',
    3: 'At-Risk Group'
}

df_clean['Cluster_Name'] = df_clean['Cluster'].map(cluster_labels)

print('Cluster Summary:')
display(df_clean['Cluster_Name'].value_counts().rename('Count').to_frame())

In [ ]:
colors = {0: '#f39c12', 1: '#3498db', 2: '#2ecc71', 3: '#e74c3c'}

plt.figure(figsize=(11, 7))

for cluster_id, label in cluster_labels.items():
    mask = kmeans.labels_ == cluster_id
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        label=f'Cluster {cluster_id}: {label}',
        alpha=0.5, s=15,
        color=colors[cluster_id]
    )

plt.title('Student Segments — K-Means (PCA 2D View, Labeled)', fontsize=14)
plt.xlabel(f'PC1 ({explained[0]*100:.1f}% Variance Explained)')
plt.ylabel(f'PC2 ({explained[1]*100:.1f}% Variance Explained)')
plt.legend(loc='upper right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Business Insights & Actionable Recommendations

Clustering 15,000 student social network profiles produced four distinct student personas.  
Below are tailored recommendations for school administrators, counsellors, and policymakers.

---

### 👗 Cluster 0 — The Fashion-Forward (3%)
**Profile:** Very high mentions of `hollister`, `abercrombie`, `shopping`, `mall`, `clothes`, `hair`, `cute`.

**Insight:** This small but distinct group's identity is strongly tied to consumer brands and appearance. While not a risk segment, they may be susceptible to peer pressure around material image and social validation through appearance.

**Recommendations:**
- Offer design thinking, fashion technology, or business/marketing electives to channel their interest productively.
- School counsellors should monitor for self-esteem or body-image concerns.
- Media literacy programs on advertising influence and brand culture can be valuable for this group.
- Entrepreneurship programs (e.g., start a school fashion blog or clothing drive) can redirect brand interest into creative output.

---

### 🤫 Cluster 1 — The Silent Majority (70%)
**Profile:** Very low keyword mention counts across all categories. These students are barely active on their social profiles.

**Insight:** A large disengaged student population online does not mean they are disengaged in school — but it can signal introversion, lack of platform access, or disinterest in social networks. Being "invisible" online means data-driven welfare programs will miss them entirely without proactive outreach.

**Recommendations:**
- Survey this group to understand whether the silence is by choice or due to digital access barriers.
- Use offline community-building approaches (clubs, mentorship) to reach this majority.
- Offer structured digital literacy and online safety programs so when they do engage, it is positive.
- Do not neglect them — their lack of online signal makes them invisible to data-driven interventions.

---

### 🏈🎵 Cluster 2 — The Actively Engaged (23%)
**Profile:** Elevated mentions of sports (basketball, football), music, dance, god, church, shopping.

**Insight:** These students are the most well-rounded and constructively engaged group. They balance physical activity, cultural interests, and faith. They are the natural leaders and ambassadors of school culture.

**Recommendations:**
- Invest in sports programs, school bands, and performing arts to keep this group thriving.
- Identify natural leaders here for peer mentoring programs targeting the Silent Majority.
- Offer career-path programs connecting their interests: sports science, music technology, community leadership.
- Leverage their high engagement for school events and community service initiatives.

---

### ⚠️ Cluster 3 — The At-Risk Group (4%)
**Profile:** Very high mentions of `drunk`, `drugs`, `die`, `death`, alongside high `sex`, `kissed`, `hair`, `music`, `rock`.

**Insight:** This is the highest-priority group from a student welfare standpoint. The combination of risky behaviour keywords with high overall activity suggests students who are both highly expressive and potentially in unsafe environments. Even at 4%, this represents **~528 students** — each of whom may need individual attention.

**Recommendations:**
- **Immediate Priority:** School counsellors and welfare officers should proactively identify and reach out to students profiling in this cluster.
- Implement targeted substance abuse prevention and mental health support programs.
- Create anonymous, safe digital channels for students to express distress without fear of judgment.
- Engage parents with awareness workshops about social media warning signs.
- Peer-led support groups can be effective here, as this group shows strong social engagement.

---

### Final Summary

| Cluster | Label | Size | Priority | Key Action |
|---|---|---|---|---|
| 0 | Fashion-Forward | ~432 | Low-Medium | Media literacy, design electives |
| 1 | Silent Majority | ~10,520 | Medium | Offline engagement, access programs |
| 2 | Actively Engaged | ~3,517 | Low | Sustain and leverage as peer leaders |
| 3 | At-Risk Group | ~528 | **HIGH** | Immediate counsellor intervention |

---

> **Takeaway:** Unsupervised learning revealed hidden student segments from raw keyword data — with no pre-defined categories.  
> The most critical business outcome is the early identification of the **At-Risk Group (Cluster 1)**, which would be nearly impossible to flag manually across 15,000 profiles.  
> This demonstrates the real-world value of K-Means clustering in education and student welfare.

---

## How to Run

```bash
pip install pandas numpy matplotlib seaborn scikit-learn scipy
jupyter notebook Student_Segmentation_KMeans.ipynb
```

The dataset `Clustering_Marketing.csv` must be in the same directory as the notebook.

---

## Project Structure

```
04_Student_Segmentation_KMeans/
├── Student_Segmentation_KMeans.ipynb   # Full analysis notebook
├── Clustering_Marketing.csv             # Dataset (15,000 student profiles)
└── README.md                            # Documentation
```